# LAB 8 — Part 1 & 2: Dataset Creation
## Domain: Procurement Compliance Q&A (Supplier Due Diligence)

**Domain choice rationale:**  
Procurement compliance is the core of the SpendLens project. Real procurement teams need to quickly extract answers from supplier contracts, GDPR data-processing agreements, and audit checklists. This dataset evaluates whether an LLM can correctly answer supplier due diligence questions — a high-stakes, verifiable domain where wrong answers have real business consequences.

**Dataset structure:**
- `input.question` — a due-diligence question a procurement analyst would ask
- `input.context` — a short supplier document excerpt (contract clause, policy snippet, audit note)
- `output.answer` — the ground-truth answer (for evaluator reference)
- `metadata.category` — question type: risk, compliance, data-privacy, contractual, financial
- `metadata.difficulty` — easy / medium / hard


## Step 1: Install dependencies

In [1]:
%pip install langsmith openai openevals python-dotenv pandas --quiet

Note: you may need to restart the kernel to use updated packages.


C:\Users\eugnm\OneDrive\Desktop\ironhack\Week 6\LAB 8\.venv\Scripts\python.exe: No module named pip


## Step 2: Environment setup & LangSmith client

In [2]:
import os
from dotenv import load_dotenv

# Load keys from .env in the same directory
load_dotenv(dotenv_path=".env", override=True)

# Verify all required env vars are present
required_vars = [
    "OPENAI_API_KEY",
    "LANGCHAIN_API_KEY",
    "LANGCHAIN_TRACING_V2",
    "LANGCHAIN_ENDPOINT",
    "LANGCHAIN_PROJECT",
]
missing = [v for v in required_vars if not os.getenv(v) or "your" in (os.getenv(v) or "")]
if missing:
    raise EnvironmentError(
        f"\n❌ Missing or unfilled variables in .env: {missing}\n"
        "→ Open .env and fill in your API keys before continuing."
    )

print("✅ Environment loaded")
print(f"   Project : {os.getenv('LANGCHAIN_PROJECT')}")
print(f"   Endpoint: {os.getenv('LANGCHAIN_ENDPOINT')}")
print(f"   Tracing : {os.getenv('LANGCHAIN_TRACING_V2')}")

✅ Environment loaded
   Project : procurement-compliance-eval
   Endpoint: https://api.smith.langchain.com
   Tracing : true


In [3]:
from langsmith import Client

# Initialize LangSmith client
ls_client = Client(
    api_url=os.getenv("LANGCHAIN_ENDPOINT"),
    api_key=os.getenv("LANGCHAIN_API_KEY"),
)

# Quick connectivity check
try:
    projects = list(ls_client.list_projects())
    print(f"✅ LangSmith connected — {len(projects)} project(s) visible")
except Exception as e:
    print(f"❌ LangSmith connection failed: {e}")
    print("   → Check LANGCHAIN_API_KEY and LANGCHAIN_ENDPOINT in .env")

✅ LangSmith connected — 1 project(s) visible


## Step 3: Define the procurement compliance dataset

20 hand-crafted examples across five procurement compliance categories.  
Each example mirrors real questions a SpendLens / supplier-DD workflow would face.

In [4]:
# ── Raw examples ──────────────────────────────────────────────────────────────
# Format: (question, context_excerpt, ground_truth_answer, category, difficulty)

RAW_EXAMPLES = [
    # ── RISK ──────────────────────────────────────────────────────────────────
    (
        "What is the supplier's maximum liability cap under the contract?",
        "Section 12.3 — Limitation of Liability: The total aggregate liability of Supplier "
        "to Buyer under or in connection with this Agreement, whether in contract, tort "
        "(including negligence), breach of statutory duty or otherwise, shall not exceed "
        "the total Fees paid or payable by Buyer in the twelve (12) months immediately "
        "preceding the event giving rise to the claim.",
        "The supplier's maximum liability is capped at the total fees paid by the buyer "
        "in the 12 months before the event giving rise to the claim.",
        "risk",
        "easy",
    ),
    (
        "Does the contract allow the supplier to subcontract without buyer approval?",
        "Section 8.1 — Subcontracting: Supplier shall not subcontract any part of the "
        "Services without the prior written consent of Buyer, such consent not to be "
        "unreasonably withheld or delayed. Any approved subcontractor shall be bound by "
        "obligations no less onerous than those in this Agreement.",
        "No. The supplier must obtain prior written consent from the buyer before "
        "subcontracting any part of the services.",
        "risk",
        "easy",
    ),
    (
        "What happens if the supplier fails to meet the SLA uptime target?",
        "Schedule 3 — SLA: Supplier shall maintain 99.5% monthly uptime. For each full "
        "percentage point below 99.5%, Buyer shall receive a service credit equal to 5% "
        "of the monthly fee for that service, up to a maximum credit of 25% of the "
        "monthly fee. Credits are the sole remedy for SLA breaches.",
        "For each full percentage point below the 99.5% target, the buyer receives a "
        "service credit of 5% of the monthly fee, capped at 25%. Credits are the "
        "exclusive remedy for SLA failures.",
        "risk",
        "medium",
    ),
    (
        "Is the supplier required to maintain cyber insurance?",
        "Section 15 — Insurance: Supplier shall, at its own cost, maintain throughout the "
        "Term: (a) commercial general liability insurance with a limit of no less than "
        "EUR 2,000,000 per occurrence; (b) cyber liability and data breach insurance with "
        "a minimum limit of EUR 5,000,000 per claim; (c) professional indemnity insurance "
        "with a minimum limit of EUR 3,000,000 per claim. Certificates of insurance shall "
        "be provided to Buyer upon request.",
        "Yes. The supplier must maintain cyber liability and data breach insurance with a "
        "minimum limit of EUR 5,000,000 per claim for the full contract term.",
        "risk",
        "medium",
    ),
    # ── COMPLIANCE ────────────────────────────────────────────────────────────
    (
        "Is the supplier required to comply with anti-bribery laws?",
        "Section 19 — Anti-Bribery and Corruption: Supplier shall comply with all "
        "applicable anti-bribery and anti-corruption laws and regulations, including the "
        "UK Bribery Act 2010 and the US Foreign Corrupt Practices Act, and shall not "
        "offer, pay, or receive any bribe, kickback, or improper payment in connection "
        "with this Agreement. Breach of this clause is a material breach entitling Buyer "
        "to terminate immediately.",
        "Yes. The supplier must comply with anti-bribery laws including the UK Bribery "
        "Act 2010 and US FCPA. A breach is treated as a material breach allowing "
        "immediate termination.",
        "compliance",
        "easy",
    ),
    (
        "What audit rights does the buyer have over the supplier?",
        "Section 20 — Audit Rights: Buyer (or its appointed third-party auditor) shall "
        "have the right, on 10 Business Days' written notice, to audit Supplier's "
        "books, records, systems, and facilities relevant to the Services and this "
        "Agreement, no more than once per calendar year unless a material breach is "
        "reasonably suspected. Supplier shall provide reasonable cooperation and access.",
        "The buyer can audit the supplier's records, systems, and facilities once per "
        "year with 10 business days' notice, or more frequently if a material breach is "
        "suspected. The supplier must cooperate.",
        "compliance",
        "medium",
    ),
    (
        "Is the supplier obligated to report ethical violations in its supply chain?",
        "Supplier Code of Conduct, Clause 7 — Supply Chain Transparency: Supplier shall "
        "conduct reasonable due diligence on its own sub-suppliers and shall promptly "
        "notify Buyer of any identified or suspected violations of human rights, forced "
        "labour, child labour, or environmental regulations within its supply chain. "
        "Notification must occur within five (5) Business Days of Supplier becoming "
        "aware of the violation.",
        "Yes. The supplier must notify the buyer within 5 business days of becoming "
        "aware of human rights, forced/child labour, or environmental violations in "
        "its own supply chain.",
        "compliance",
        "medium",
    ),
    (
        "What is the notice period required for contract termination for convenience?",
        "Section 22.2 — Termination for Convenience: Either party may terminate this "
        "Agreement without cause by providing the other party with not less than "
        "ninety (90) days' written notice. Upon expiry of such notice, Buyer shall pay "
        "all undisputed Fees for Services rendered up to the termination date.",
        "Either party must provide 90 days' written notice to terminate the contract "
        "for convenience. The buyer must pay undisputed fees for services rendered "
        "up to the termination date.",
        "compliance",
        "easy",
    ),
    # ── DATA PRIVACY ──────────────────────────────────────────────────────────
    (
        "Is the supplier acting as a data processor or data controller under GDPR?",
        "Data Processing Agreement (DPA), Clause 2 — Roles: The parties acknowledge "
        "that for the purposes of the GDPR, Buyer is the Data Controller and Supplier "
        "is the Data Processor with respect to any Personal Data processed by Supplier "
        "in the course of providing the Services. Supplier shall process Personal Data "
        "only on documented instructions from Buyer.",
        "The supplier is a Data Processor and the buyer is the Data Controller under "
        "GDPR. The supplier may only process personal data on documented instructions "
        "from the buyer.",
        "data-privacy",
        "easy",
    ),
    (
        "Where is the supplier allowed to store and process personal data?",
        "DPA Clause 8 — Data Transfers: Supplier shall process and store all Personal "
        "Data exclusively within the European Economic Area (EEA), unless Buyer provides "
        "prior written consent for transfer to a third country. Any transfer outside the "
        "EEA must be governed by Standard Contractual Clauses (SCCs) approved by the "
        "European Commission.",
        "Personal data must be stored and processed exclusively within the EEA. "
        "Transfers outside the EEA require buyer's prior written consent and must be "
        "covered by EU Standard Contractual Clauses.",
        "data-privacy",
        "medium",
    ),
    (
        "How long can the supplier retain personal data after contract termination?",
        "DPA Clause 11 — Retention and Deletion: Upon termination or expiry of the "
        "Agreement, Supplier shall, at Buyer's choice, delete or return all Personal "
        "Data within thirty (30) days. Supplier may retain Personal Data for a further "
        "period only where required to do so by applicable law, in which case Supplier "
        "shall inform Buyer of such legal requirement prior to retention.",
        "The supplier must delete or return all personal data within 30 days of contract "
        "termination. Longer retention is only permitted if required by law, and the "
        "supplier must inform the buyer in advance.",
        "data-privacy",
        "medium",
    ),
    (
        "What must the supplier do if it experiences a personal data breach?",
        "DPA Clause 13 — Data Breach Notification: Supplier shall notify Buyer without "
        "undue delay and, where feasible, no later than twenty-four (24) hours after "
        "becoming aware of a Personal Data Breach. The notification shall include the "
        "nature of the breach, categories and approximate number of data subjects "
        "affected, likely consequences, and measures taken or proposed.",
        "The supplier must notify the buyer within 24 hours of becoming aware of a data "
        "breach, including details of the breach's nature, affected data subjects, "
        "consequences, and remediation measures taken.",
        "data-privacy",
        "hard",
    ),
    (
        "Can the supplier appoint sub-processors without the buyer's knowledge?",
        "DPA Clause 9 — Sub-processors: Supplier shall not engage any new sub-processor "
        "without giving Buyer at least 14 days' prior written notice. Buyer shall have "
        "the right to object to the new sub-processor within that period. If Buyer "
        "objects and the parties cannot resolve the disagreement, Buyer may terminate "
        "the Agreement without penalty.",
        "No. The supplier must give the buyer at least 14 days' written notice before "
        "appointing a new sub-processor. The buyer can object, and if unresolved, "
        "may terminate without penalty.",
        "data-privacy",
        "hard",
    ),
    # ── CONTRACTUAL ───────────────────────────────────────────────────────────
    (
        "What is the contract's governing law and jurisdiction?",
        "Section 26 — Governing Law: This Agreement and any dispute or claim arising "
        "out of or in connection with it shall be governed by and construed in accordance "
        "with the laws of England and Wales. Each party irrevocably submits to the "
        "exclusive jurisdiction of the courts of England and Wales.",
        "The contract is governed by the laws of England and Wales. Any disputes are "
        "subject to the exclusive jurisdiction of English and Welsh courts.",
        "contractual",
        "easy",
    ),
    (
        "Does the contract include an automatic renewal clause?",
        "Section 3.2 — Renewal: Unless either party provides written notice of "
        "non-renewal at least sixty (60) days before the end of the then-current Term, "
        "this Agreement shall automatically renew for successive one-year periods on "
        "the same terms and conditions, subject to any price adjustment per Section 6.4.",
        "Yes. The contract auto-renews for successive one-year periods unless either "
        "party gives 60 days' written notice of non-renewal before the current term ends.",
        "contractual",
        "medium",
    ),
    (
        "Can the supplier unilaterally change the pricing during the contract term?",
        "Section 6.4 — Price Adjustments: Supplier may increase Fees at the start of "
        "each renewal term by no more than the percentage change in the Consumer Price "
        "Index (CPI) for the preceding 12-month period, provided that Supplier gives "
        "Buyer not less than 90 days' written notice of any such increase prior to the "
        "renewal date.",
        "The supplier can increase fees at renewal only, capped at the CPI increase for "
        "the prior 12 months, with 90 days' advance written notice. No mid-term "
        "price changes are permitted.",
        "contractual",
        "hard",
    ),
    (
        "What intellectual property does the buyer own from work produced under the contract?",
        "Section 14 — Intellectual Property: All deliverables, reports, software, and "
        "other materials created specifically for Buyer under this Agreement "
        "('Buyer Works') shall be works made for hire and the exclusive property of "
        "Buyer upon creation. Supplier hereby assigns all right, title, and interest "
        "in Buyer Works to Buyer. Supplier retains ownership of its pre-existing "
        "intellectual property ('Supplier Background IP').",
        "The buyer owns all deliverables and materials created specifically under the "
        "contract (works made for hire). The supplier retains its pre-existing "
        "background IP.",
        "contractual",
        "medium",
    ),
    # ── FINANCIAL ─────────────────────────────────────────────────────────────
    (
        "What are the payment terms under the contract?",
        "Section 6.1 — Payment Terms: Supplier shall issue invoices monthly in arrears. "
        "Buyer shall pay each undisputed invoice within thirty (30) days of receipt. "
        "Late payments shall accrue interest at the rate of 2% above the Bank of England "
        "base rate per annum, calculated daily from the due date until actual payment.",
        "Invoices are issued monthly in arrears and must be paid within 30 days. "
        "Late payments accrue interest at 2% above the Bank of England base rate "
        "per annum.",
        "financial",
        "easy",
    ),
    (
        "Is the buyer allowed to withhold payment for disputed invoices?",
        "Section 6.3 — Disputed Invoices: Buyer may withhold payment of any amount "
        "genuinely disputed in good faith, provided that Buyer: (a) pays all "
        "undisputed amounts by the due date; (b) notifies Supplier in writing of the "
        "disputed amount and reasons within 10 Business Days of receipt of the invoice; "
        "and (c) cooperates in good faith to resolve the dispute promptly.",
        "Yes. The buyer may withhold genuinely disputed amounts if it pays undisputed "
        "amounts on time, notifies the supplier in writing within 10 business days, "
        "and cooperates to resolve the dispute.",
        "financial",
        "medium",
    ),
    (
        "Does the contract include a most-favoured-nation (MFN) pricing clause?",
        "Section 6.5 — Most-Favoured Pricing: Supplier warrants that the Fees charged "
        "to Buyer are no greater than those charged to any other customer of Supplier "
        "for substantially similar services and volumes. If Supplier offers lower fees "
        "to another customer, Supplier shall immediately notify Buyer and apply "
        "equivalent pricing to Buyer's account.",
        "Yes. The contract includes an MFN clause: if the supplier offers lower fees "
        "to any other customer for similar services, it must immediately apply those "
        "lower fees to the buyer.",
        "financial",
        "hard",
    ),
]

print(f"✅ {len(RAW_EXAMPLES)} examples defined across 5 categories")

✅ 20 examples defined across 5 categories


In [5]:
import pandas as pd

# Convert to structured format for inspection
rows = []
for q, ctx, ans, cat, diff in RAW_EXAMPLES:
    rows.append(
        {
            "question": q,
            "context": ctx[:80] + "...",  # truncated for display
            "answer": ans[:80] + "...",
            "category": cat,
            "difficulty": diff,
        }
    )

df = pd.DataFrame(rows)
print(df.groupby(["category", "difficulty"]).size().to_string())
print(f"\nTotal examples: {len(df)}")

category      difficulty
compliance    easy          2
              medium        2
contractual   easy          1
              hard          1
              medium        2
data-privacy  easy          1
              hard          2
              medium        2
financial     easy          1
              hard          1
              medium        1
risk          easy          2
              medium        2



Total examples: 20


## Step 4: Create LangSmith dataset

Checks if the dataset already exists (idempotent — safe to re-run).

In [6]:
DATASET_NAME = "procurement-compliance-qa-v1"
DATASET_DESCRIPTION = (
    "Procurement compliance Q&A dataset. Each example contains a due-diligence "
    "question, a supplier document excerpt as context, and a ground-truth answer. "
    "Categories: risk, compliance, data-privacy, contractual, financial. "
    "20 examples, difficulty: easy / medium / hard. "
    "Designed to evaluate LLM contract-comprehension and compliance-reasoning quality."
)

# Check if dataset already exists
existing_datasets = {d.name for d in ls_client.list_datasets()}

if DATASET_NAME in existing_datasets:
    print(f"ℹ️  Dataset '{DATASET_NAME}' already exists — fetching it")
    dataset = ls_client.read_dataset(dataset_name=DATASET_NAME)
else:
    print(f"Creating dataset '{DATASET_NAME}'...")
    dataset = ls_client.create_dataset(
        dataset_name=DATASET_NAME,
        description=DATASET_DESCRIPTION,
    )
    print(f"✅ Dataset created: {dataset.id}")

print(f"   Dataset URL: https://smith.langchain.com/datasets/{dataset.id}")

ℹ️  Dataset 'procurement-compliance-qa-v1' already exists — fetching it
   Dataset URL: https://smith.langchain.com/datasets/cd1896d2-e141-46d7-870a-27bf37a91ce5


In [7]:
# Upload examples (only if dataset was freshly created or is empty)
existing_examples = list(ls_client.list_examples(dataset_id=dataset.id))

if len(existing_examples) >= len(RAW_EXAMPLES):
    print(f"ℹ️  Dataset already has {len(existing_examples)} examples — skipping upload")
else:
    print(f"Uploading {len(RAW_EXAMPLES)} examples...")
    for q, ctx, ans, cat, diff in RAW_EXAMPLES:
        ls_client.create_example(
            inputs={"question": q, "context": ctx},
            outputs={"answer": ans},
            metadata={"category": cat, "difficulty": diff},
            dataset_id=dataset.id,
        )
    print(f"✅ {len(RAW_EXAMPLES)} examples uploaded")

# Final count
total = len(list(ls_client.list_examples(dataset_id=dataset.id)))
print(f"\nDataset '{DATASET_NAME}' now contains {total} examples")

ℹ️  Dataset already has 20 examples — skipping upload

Dataset 'procurement-compliance-qa-v1' now contains 20 examples


## Step 5: Save dataset locally as JSON backup

In [8]:
import json

local_dataset = [
    {
        "inputs": {"question": q, "context": ctx},
        "outputs": {"answer": ans},
        "metadata": {"category": cat, "difficulty": diff},
    }
    for q, ctx, ans, cat, diff in RAW_EXAMPLES
]

with open("data/procurement_compliance_dataset.json", "w", encoding="utf-8") as f:
    json.dump(local_dataset, f, indent=2, ensure_ascii=False)

print("✅ Local backup saved to data/procurement_compliance_dataset.json")
print(f"   {len(local_dataset)} examples | Dataset name: {DATASET_NAME}")

✅ Local backup saved to data/procurement_compliance_dataset.json
   20 examples | Dataset name: procurement-compliance-qa-v1


## ✅ Part 1 & 2 Complete

| Item | Status |
|------|--------|
| Domain chosen | Procurement Compliance Q&A |
| Examples created | 20 (5 categories × 4 questions each) |
| LangSmith dataset | `procurement-compliance-qa-v1` |
| Local backup | `data/procurement_compliance_dataset.json` |

**Next:** Run `02_evaluation.ipynb` to implement the target function and evaluators.